# CelerisAi 1D Tutorial

This notebook reproduces the `setrun_1D.py` example and explains each setup step.

What you will do:
- Initialize Taichi
- Load 1D topography and wave forcing
- Build the domain and solver
- Run the model in display mode (or headless mode)

## Prerequisites
If you have not installed CelerisAi yet, you can do it just with:

From repository root:
```bash
pip install -e .
```

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

## 1) Imports and backend

In [ ]:
import taichi as ti

from celeris.domain import Topodata, BoundaryConditions, Domain
from celeris.solver import Solver
from celeris.runner import Evolve

# Use ti.cpu if you do not want GPU or need easier debugging
ti.init(arch=ti.gpu)

## 2) Define input data

This matches `setrun_1D.py` and uses files under `examples/1D/`.

In [ ]:
EXAMPLE_DIR = str(repo_root / 'examples' / '1D')

baty = Topodata(
    filename='Topo1D.txt',
    path=EXAMPLE_DIR,
    datatype='xz'
)

bc = BoundaryConditions(
    West=2,
    celeris=False,
    path=EXAMPLE_DIR,
    filename='irrWaves1D.txt'
)

EXAMPLE_DIR

## 3) Build domain and solver

In [ ]:
d = Domain(
    topodata=baty,
    x1=0.0,
    x2=480.0,
    Nx=480
)

solver = Solver(
    model='Bouss',
    domain=d,
    boundary_conditions=bc,
    timeScheme=2,
    pred_or_corrector=True,
    useBreakingModel=True
)

solver.nx, solver.ny, solver.dt

## 4) Run model

`plot_interval` controls how often logs and (if enabled) images are produced.

In [ ]:
run = Evolve(solver=solver, maxsteps=3000, saveimg=False, plot_interval=100)

# Interactive 1D display
#run.Evolve_1D_Display()

# Headless alternative:
run.Evolve_Headless()

## Notes

- If your machine has no CUDA/compatible GPU backend, switch to `ti.init(arch=ti.cpu)`.
- For faster test runs, reduce `maxsteps`.
- To save snapshots/GIF inputs, set `saveimg=True` and tune `plot_interval`.

## 5 Plot the results (last step)
This process can be done in any step of the numerical simulation

In [ ]:
import matplotlib.pyplot as plt
# extract the coordinates
x_coord,zz =solver.domain.topofield()

#Send data to numpy
q = run.solver.State.to_numpy()
eta = q[:,0,0]
hu = q[:,0,1]
plt.plot(x_coord,eta,color='b')
plt.plot(x_coord,-zz,color='k')# make bathymetry negative